In [1]:
import sys
import os
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(project_root)
from datetime import datetime

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import HashingVectorizer, TfidfVectorizer
import joblib
from tqdm import tqdm
from concurrent.futures import as_completed
from common.const import (
    SCHEMA,
    TABLE_RAW_REVIEWS,
    TABLE_FEATURED_USER_VECTORS,
    TABLE_USER_AGGREGATED_STATE,
    TABLE_MATERIALIZED_VIEW_USER,
    TABLE_USER_ITEM_STATE,          
    TABLE_USER_ITEM_STATE_SNAPSHOT,
    TABLE_FEATURED_DENSE_VECTORS,   
    TABLE_VECTOR_PRODUCTS
)
from common.helper import connect_clickhouse, clickhouse_extract_data_as_df
from concurrent.futures import ThreadPoolExecutor
import threading

c:\Users\Admin\.conda\envs\scc454\lib\site-packages\torchvision\io\image.py:13: UserWarning: Failed to load image Python extension: '[WinError 127] The specified procedure could not be found'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


In [2]:
VECTOR_DIM = 387

In [3]:
def create_tables():
    create_table_query = f"""
    CREATE TABLE IF NOT EXISTS {SCHEMA}.{TABLE_VECTOR_PRODUCTS} (
        id           UInt64,
        parent_asin  String,
        mixed_vector Array(Float32) CODEC(NONE),
        INDEX bloom_asin parent_asin TYPE bloom_filter(0.01) GRANULARITY 1,
        INDEX hnsw_idx mixed_vector
            TYPE vector_similarity(
                'hnsw',
                'cosineDistance',
                {VECTOR_DIM},
                'bf16',
                32,
                200
            )
            GRANULARITY 1
    ) ENGINE = MergeTree
    ORDER BY id
    PRIMARY KEY id;
    """

    populate_query = f"""
    INSERT INTO {SCHEMA}.{TABLE_VECTOR_PRODUCTS}
    SELECT
        rowNumberInAllBlocks() AS id,
        parent_asin,
        mixed_vector
    FROM {SCHEMA}.{TABLE_FEATURED_DENSE_VECTORS}
    ORDER BY parent_asin;
    """

    client = connect_clickhouse()
    try:
        client.command(create_table_query)
        client.command(populate_query)
    finally:
        client.close()
create_tables()